## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [4]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [5]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [6]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [7]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [8]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [ ]:
# Write collection_bbox(features) and apply to all four LOD files
# Your code here

In [10]:
import json
from pathlib import Path

lod_dir = Path("../../data/lod")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

lod_data = {}
for name, filename in lod_files.items():
    path = lod_dir / filename
    with open(path) as f:
        lod_data[name] = json.load(f)
    print(f"Loaded {name}: {len(lod_data[name]['features']):,} features")

Loaded coarse: 25,413 features
Loaded medium: 25,413 features
Loaded fine: 25,413 features
Loaded extra_fine: 25,413 features


In [14]:
import json

def collection_bbox(geojson_data):
    """
    Calculates the bounding box [minX, minY, maxX, maxY]
    of an entire FeatureCollection.
    """
    min_x, min_y = float('inf'), float('inf')
    max_x, max_y = float('-inf'), float('-inf')

    # Iterate through each feature
    for feature in geojson_data['features']:
        geom = feature['geometry']
        coords = []

        # Handle different geometry types (simplification: assumes LineString/Polygon/Point)
        if geom['type'] == 'Point':
            coords = [geom['coordinates']]
        elif geom['type'] in ['LineString', 'MultiPoint']:
            coords = geom['coordinates']
        elif geom['type'] in ['Polygon', 'MultiLineString']:
            # Flatten polygon rings
            for ring in geom['coordinates']:
                coords.extend(ring)
        elif geom['type'] == 'MultiPolygon':
            for polygon in geom['coordinates']:
                for ring in polygon:
                    coords.extend(ring)

        # Update global bounds
        for coord in coords:
            x, y = coord[0], coord[1]
            if x < min_x: min_x = x
            if y < min_y: min_y = y
            if x > max_x: max_x = x
            if y > max_y: max_y = y

    return [min_x, min_y, max_x, max_y]

# Apply collection_bbox to all LOD files already loaded
results = {}
for name, data in lod_data.items():
    bbox = collection_bbox(data)
    results[name] = bbox
    print(f"{name}: {bbox}")


coarse: [-149.4175, -51.636701, 179.357778, 69.604375]
medium: [-150.112222, -51.894722, 179.357778, 69.604375]
fine: [-150.112222, -51.894722, 179.357778, 69.604375]
extra_fine: [-150.112222, -51.895278, 179.357778, 69.604375]


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [20]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, Rectangle, Marker, basemaps

# 1. Use the already loaded fine LOD data (from previous cells)
fine = lod_data['fine']

# 2. Calculate Bounding Box Area for each feature
features_with_area = []
for feature in fine['features']:
    coords = feature['geometry']['coordinates']
    
    # Extract lons/lats based on geometry type (railroads are LineStrings)
    if feature['geometry']['type'] == 'LineString':
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
    else:
        continue  # Skip other types if any

    lon_min, lon_max = min(lons), max(lons)
    lat_min, lat_max = min(lats), max(lats)
    
    area = (lon_max - lon_min) * (lat_max - lat_min)
    features_with_area.append({
        'feature': feature,
        'area': area,
        'category': feature['properties'].get('category', 'Unknown'),
        'bounds': ((lat_min, lon_min), (lat_max, lon_max))
    })

# 3. Sort by area descending and select top 5
top_5_features = sorted(features_with_area, key=lambda x: x['area'], reverse=True)[:5]

# 4. Print Results
print("Top 5 Largest Features by BBox Area:")
for i, feat in enumerate(top_5_features):
    print(f"{i+1}. Category: {feat['category']}, Area: {feat['area']:.4f}")

# 5. Visualize on ipyleaflet
m = Map(center=[20, 0], zoom=3, basemap=basemaps.Esri.WorldImagery)
for feat in top_5_features:
    # Add rectangle for bbox
    rectangle = Rectangle(bounds=feat['bounds'], color='red', fill_opacity=0.8)
    m.add_layer(rectangle)
    
    # Add line itself
    geo_json_layer = GeoJSON(data=feat['feature'], style={'color': 'blue', 'weight': 1})
    m.add_layer(geo_json_layer)

m

Top 5 Largest Features by BBox Area:
1. Category: 0, Area: 29.3123
2. Category: 0, Area: 18.6487
3. Category: 3, Area: 17.7045
4. Category: 2, Area: 17.3866
5. Category: 2, Area: 12.3547


Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [ ]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson
# Your code here

## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

Two different railroad features can have identical bounding boxes despite completely different paths, typically occurring when comparing a straight rail segment with a complex, small-radius, or "curled" path that occupies the same maximum spatial extent.

If the straight track (Feature A) is supposed to be fully hidden behind a mountain, but the loop (Feature B) actually crosses into the view behind the mountain, the system might incorrectly cull both if it assumes they have the same location, or fail to cull either.


## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.